# FINLORA PLUMBING 
*Description :*
Notebook to explore data and make data cleaning decisions and select meaning EDA.

In [327]:
import pandas as pd 
import numpy as np

In [328]:
df_raw = pd.read_csv("../data_assets/Finlora_Customer_Transaction_Dataset.csv")
df_raw.shape

(11400, 26)

In [329]:
df_raw.columns

Index(['transaction_id', 'customer_id', 'timestamp', 'home_country',
       'source_currency', 'dest_currency', 'channel', 'amount_src',
       'amount_usd', 'fee', 'exchange_rate_src_to_dest', 'device_id',
       'new_device', 'ip_address', 'ip_country', 'location_mismatch',
       'ip_risk_score', 'kyc_tier', 'account_age_days', 'device_trust_score',
       'chargeback_history_count', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'is_fraud'],
      dtype='str')

In [330]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  str    
 1   customer_id                11400 non-null  str    
 2   timestamp                  11371 non-null  str    
 3   home_country               11400 non-null  str    
 4   source_currency            11400 non-null  str    
 5   dest_currency              11400 non-null  str    
 6   channel                    11400 non-null  str    
 7   amount_src                 11400 non-null  str    
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  str    
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  str    
 14  i

In [331]:
df_raw.describe()

,amount_usd,fee,exchange_rate_src_to_dest,ip_risk_score,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
count,11095.000000,11105.000000,11400.000000,11400.000000,11400.000000,11105.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000
mean,452.022083,100.309441,167.540397,0.396726,393.793158,0.653681,0.048509,0.267134,0.458333,0.723509,0.045501,0.087456
std,1403.973062,958.128504,382.023827,0.270507,342.348393,0.273012,0.256194,0.142983,1.524494,1.958390,0.084942,0.282515
min,7.230000,-1.000000,0.592000,0.004000,1.000000,-0.100000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
25%,92.465000,2.380000,1.000000,0.209000,147.000000,0.515000,0.000000,0.169000,0.000000,0.000000,0.000000,0.000000
50%,163.480000,3.500000,7.142857,0.325000,298.000000,0.658000,0.000000,0.223000,0.000000,0.000000,0.000000,0.000000
75%,302.190000,5.550000,73.529412,0.487000,661.000000,0.894000,0.000000,0.391000,0.000000,0.000000,0.050000,0.000000
max,12498.570000,9999.990000,1388.888889,1.200000,1095.000000,0.999000,2.000000,0.900000,8.000000,11.000000,0.250000,1.000000


In [332]:
# I display a redacted sample here so no real IDs land in saved outputs.
# show() masks customer/transaction/device/IP identifiers with stable
# labels, on a throwaway copy only — df_raw itself is untouched.
# (helper lives in ../redact.py; my exploration below uses the real frame.)
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(".."))
from redact import show
show(df_raw, 50)

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,exchange_rate_src_to_dest,device_id,new_device,ip_address,ip_country,location_mismatch,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,txn_001,cust_001,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,1.351351,dev_001,False,ip_001,US,False,0.123,standard,263,0.522,0,0.223,0,0,0.00,0
1,txn_002,cust_002,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,12.758621,dev_002,True,ip_002,CA,False,0.569,standard,947,0.475,0,0.268,0,1,0.00,0
2,txn_003,cust_003,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,7.142857,dev_003,False,ip_003,US,False,0.437,enhanced,367,0.939,0,0.176,0,0,0.00,0
3,txn_004,cust_004,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,0.925926,dev_004,False,ip_004,US,False,0.594,standard,147,0.551,0,0.391,0,0,0.00,0
4,txn_005,cust_005,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,83.333333,dev_005,False,ip_005,US,False,0.121,enhanced,257,0.894,0,0.257,0,0,0.00,0
5,txn_006,cust_006,2022-10-04 12:09:59.468549+00:00,US,USD,GBP,mobile,526.9,526.90,8.75,0.800000,dev_006,False,ip_006,US,False,0.094,low,616,0.702,0,0.361,0,0,0.00,0
6,txn_007,cust_002,2022-10-04 12:37:41.468549+00:00,CA,CAD,GBP,mobile,149.24,110.44,2.33,0.592000,dev_007,False,ip_007,CA,False,0.299,standard,947,0.625,0,0.268,0,0,0.00,0
7,txn_008,cust_007,2022-10-04 16:27:44.468549+00:00,CA,CAD,INR,web,276.51,204.62,5.59,61.666667,dev_008,False,ip_008,CA,False,0.087,standard,1041,0.867,0,0.245,0,1,0.12,0
8,txn_009,cust_003,2022-10-04 21:00:36.468549+00:00,US,USD,EUR,mobile,99.52,99.52,2.41,0.925926,dev_009,False,ip_009,US,False,0.182,enhanced,367,0.939,0,0.176,0,0,0.00,0
9,txn_010,cust_008,2022-10-04 21:20:42.468549+00:00,US,USD,PHP,web,302.55,302.55,5.26,58.823529,dev_010,False,ip_010,US,False,0.413,standard,1016,0.944,0,0.247,0,0,0.10,1


In [333]:
#check transaction_id for duplicates
total_unique_transactions = df_raw.transaction_id.nunique()
total_transactions = len(df_raw.transaction_id)
print(f"Total unique transactions: {total_unique_transactions}")
print(f"Total transactions: {total_transactions}")

# Are the duplicate transaction_ids full-row identical, or conflicting?
dup_id_mask = df_raw.transaction_id .duplicated(keep=False)   # every row involved in any id dupe
n_dup_rows  = dup_id_mask.sum()                                 # 400 rows
n_dup_ids   = df_raw.loc[dup_id_mask, 'transaction_id'].nunique()  # 200 distinct ids
print(f"{n_dup_rows} rows share {n_dup_ids} duplicated ids")

# Within each id-group: is every column identical, or do any columns disagree?
dup_groups = df_raw.loc[dup_id_mask].groupby('transaction_id')
conflicting_cols = []
for col in df_raw.columns:
    nunique_per_id = dup_groups[col].nunique()   # 1 = all same; 2+ = disagree
    if (nunique_per_id > 1).any():
        conflicting_cols.append(col)
        print(f"CONFLICT in '{col}': {int((nunique_per_id > 1).sum())} id-groups disagree")

print(f"\nfully-identical duplicate rows: {df_raw.duplicated().sum()}")
print(f"columns with conflicts across dupe ids: {conflicting_cols or 'NONE'}")
 


Total unique transactions: 11200
Total transactions: 11400
400 rows share 200 duplicated ids

fully-identical duplicate rows: 200
columns with conflicts across dupe ids: NONE


In [334]:
#drop duplicate transaction_id rows, keeping the first occurrence, and sort by transaction_id
#note: customer_id  duplicates are allowed since a customer can have multiple transactions
n_before = len(df_raw)
df_raw = (
    df_raw.drop_duplicates(subset=['transaction_id'], keep='first')
    .sort_values(by='transaction_id')
    .reset_index(drop=True)
)

#making sure the duplicates have been dropped
assert df_raw.transaction_id.duplicated().sum() == 0, "Duplicate transaction_id rows still exist"
print(f"dedup: {n_before} → {len(df_raw)} rows  "
      f"(removed {n_before - len(df_raw)} duplicate transaction_ids)")

dedup: 11400 → 11200 rows  (removed 200 duplicate transaction_ids)


In [335]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 11200 entries, 0 to 11199
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11200 non-null  str    
 1   customer_id                11200 non-null  str    
 2   timestamp                  11171 non-null  str    
 3   home_country               11200 non-null  str    
 4   source_currency            11200 non-null  str    
 5   dest_currency              11200 non-null  str    
 6   channel                    11200 non-null  str    
 7   amount_src                 11200 non-null  str    
 8   amount_usd                 10900 non-null  float64
 9   fee                        10910 non-null  float64
 10  exchange_rate_src_to_dest  11200 non-null  float64
 11  device_id                  11200 non-null  str    
 12  new_device                 11200 non-null  bool   
 13  ip_address                 10900 non-null  str    
 14  i

In [336]:
# convert amoun_src to numeric 
df_raw.amount_src= pd.to_numeric(
    df_raw.amount_src.astype("string").str.replace(",","",regex=False),
    errors="coerce"
    );
print(f"Unable to parse {df_raw['amount_src'].isna().sum()} amount_src")
df_raw.amount_src.head(20)

Unable to parse 0 amount_src


0       57.02
1      129.58
2       31.65
3      212.89
4      128.93
5      236.64
6      321.11
7       48.45
8      466.75
9       64.32
10      53.73
11      28.16
12     290.54
13     138.82
14     174.33
15      55.18
16    9990.65
17     177.26
18     164.99
19     414.59
Name: amount_src, dtype: Float64

In [337]:
#investigate  timestamp column and convert to datetime
print(f"Total missing timestamps: {df_raw.timestamp.isna().sum()}")
print(f"Total unique timestamps: {df_raw.timestamp.nunique()}")

Total missing timestamps: 29
Total unique timestamps: 11141


 Timestamp missing has to be investigated with other missing values . 
 *Theory* : possible pattern with other missing values 

In [338]:
#convert timestamp to datetime
converted_timestamps = pd.to_datetime(df_raw.timestamp, errors="coerce", format="mixed", utc=True)
timestamp_comparison = pd.DataFrame({
    "original_timestamp": df_raw.timestamp,
    "converted_timestamp": converted_timestamps,
    "invalid_timestamp": converted_timestamps.isna()
})
timestamp_comparison[timestamp_comparison["invalid_timestamp"] == True].head(29)

,original_timestamp,converted_timestamp,invalid_timestamp
60,2025/13/40 25:61:00,NaT,True
73,NaN,NaT,True
190,0000-00-00T00:00:00Z,NaT,True
668,NaN,NaT,True
831,NaN,NaT,True
1422,NaN,NaT,True
1540,NaN,NaT,True
1649,NaN,NaT,True
1715,NaN,NaT,True
1794,0000-00-00T00:00:00Z,NaT,True


In [339]:
df_raw.timestamp = converted_timestamps

In [340]:
n_bad_ts = df_raw.timestamp.isna().sum()
print(f"After parse: {len(df_raw)} rows, {n_bad_ts} unparseable timestamps")

n_missing_ts = df_raw.timestamp.isnull().sum()
print(f"Missing timestamps: {n_missing_ts}")

After parse: 11200 rows, 60 unparseable timestamps
Missing timestamps: 60


### Investigating Missing Values 

In [341]:
#which columns are missing, in what counts
missing_counts = df_raw.isnull().sum()
missing_summary = (
    missing_counts[missing_counts > 0]
    .to_frame(name="missing_count")
    .assign(missing_pct= lambda x: x.missing_count / len(df_raw) * 100).round(1)
    .sort_values(by="missing_pct", ascending=False)
)

print("Missing summary:")
print(missing_summary)


Missing summary:
                    missing_count  missing_pct
amount_usd                    300          2.7
ip_address                    300          2.7
fee                           290          2.6
ip_country                    296          2.6
kyc_tier                      295          2.6
device_trust_score            290          2.6
timestamp                      60          0.5


In [342]:
# Clustering check: do missing values overlap in the same rows?
# Purpose: when columns go missing together, they often share a root cause
# (e.g., "field not collected" batches, api failures, KYC tier gaps).
# This helps decide joint imputation vs. flag-as-feature.

missing_corr = df_raw.isnull().corr().abs()  # pairwise |r| of missingness indicators

# Build a clean long-format table of high-correlation pairs
pairs = missing_corr.where(missing_corr > 0.8).stack().reset_index()
pairs.columns = ['col_a', 'col_b', 'corr']

# Remove self-pairs and symmetric duplicates, sort by strength
pairs = pairs[
    (pairs.col_a != pairs.col_b) &
    (pairs.col_a < pairs.col_b)  # keeps only one of (a,b) / (b,a)
].sort_values('corr', ascending=False)

print("High missingness overlap (|r| > 0.8):")
if pairs.empty:
    print("  No column pairs with > 0.8 missingness correlation.")
else:
    print(pairs.head(10).to_string(index=False))

# Also show the count of rows missing multiple fields 
multi_null_counts = df_raw.isnull().sum(axis=1).value_counts().sort_index()
print("\nRows by count of missing fields:")
print(multi_null_counts.head(10))


High missingness overlap (|r| > 0.8):
             col_a              col_b     corr
        amount_usd         ip_address 1.000000
device_trust_score                fee 1.000000
        ip_country           kyc_tier 0.998264
        amount_usd         ip_country 0.993129
        ip_address         ip_country 0.993129
        amount_usd           kyc_tier 0.991404
        ip_address           kyc_tier 0.991404
        amount_usd                fee 0.982741
        amount_usd device_trust_score 0.982741
               fee         ip_address 0.982741

Rows by count of missing fields:
0    10840
1       60
4       14
5        1
6      285
Name: count, dtype: int64


In [343]:
# Analyze missingness patterns
patterns = df_raw.isnull().dot(df_raw.columns + '|')  # missingness pattern as string
pattern_counts = patterns.value_counts()
print(pattern_counts.head(10))

                                                                     10840
amount_usd|fee|ip_address|ip_country|kyc_tier|device_trust_score|      285
timestamp|                                                              60
amount_usd|ip_address|ip_country|kyc_tier|                              10
amount_usd|fee|ip_address|device_trust_score|                            4
amount_usd|fee|ip_address|ip_country|device_trust_score|                 1
Name: count, dtype: int64


In [344]:
patterns = df_raw.isnull().dot(df_raw.columns + '|')
main_cluster = patterns == 'amount_usd|fee|ip_address|ip_country|kyc_tier|device_trust_score|'

print(df_raw.loc[main_cluster, 'is_fraud'].value_counts(normalize=True))
print(df_raw.loc[~main_cluster, 'is_fraud'].value_counts(normalize=True))
print(df_raw.loc[main_cluster, 'channel'].value_counts())

is_fraud
0    0.975439
1    0.024561
Name: proportion, dtype: float64
is_fraud
0    0.909482
1    0.090518
Name: proportion, dtype: float64
channel
mobile    182
web        76
ATM        27
Name: count, dtype: int64


In [345]:
print(df_raw.groupby('is_fraud')[['risk_score_internal', 'chargeback_history_count']].mean())

          risk_score_internal  chargeback_history_count
is_fraud                                               
0                    0.240604                  0.002548
1                    0.545076                  0.529648


In [346]:
chronological_frame = df_raw.sort_values('timestamp')
# cumulative chargebacks BEFORE this row, per card/customer
chronological_frame['prior_cb'] = chronological_frame.groupby('customer_id')['chargeback_history_count'].cumsum()  # rough proxy
suspect = chronological_frame[(chronological_frame['chargeback_history_count'] > 0)]
print((suspect['chargeback_history_count'] > suspect['prior_cb']).mean())

0.0


In [347]:
g = chronological_frame.groupby('customer_id')
violations = (g['chargeback_history_count'].diff() < 0).sum()  # decreases over time?
print(violations)

0


In [348]:
# group by missingness of chargeback_history_count and calculate mean of is_fraud
print(df_raw.groupby(df_raw['chargeback_history_count'].isnull())['is_fraud'].mean())


chargeback_history_count
False    0.088839
Name: is_fraud, dtype: float64


In [349]:

chronological_frame = df_raw.sort_values('timestamp')
chronological_frame['cb_history_own'] = (
    chronological_frame.groupby('customer_id')['chargeback_history_count']  # replace with raw event flag
      .transform(lambda s: s.cumsum().shift(1).fillna(0))
)
print((chronological_frame['cb_history_own'] == chronological_frame['chargeback_history_count']).mean())

0.9609821428571429


commentary : charge_back is genuine feature signal for fraud  and not computed later after the transaction

explore missing amount_usd and imputation strategy

In [350]:
df_raw.exchange_rate_src_to_dest.dtype

dtype('float64')

In [351]:
# DISCOVERY: does exchange_rate_src_to_dest actually convert amount_src -> amount_usd?
# Hypothesis to test: amount_usd == amount_src * exchange_rate_src_to_dest
# If yes, amount_usd should equal amount_src * exchange_rate on most existing rows.

exchange_rate = df_raw.exchange_rate_src_to_dest
complete = df_raw.amount_usd.notna() & df_raw.amount_src.notna() & exchange_rate.notna()
check = df_raw.loc[complete]

pct_mult = (check.amount_usd / (check.amount_src * exchange_rate[complete])).between(0.99, 1.01)
print(f"amount_usd == src * rate  on {pct_mult.mean():.1%} of rows")

pct_div = (check.amount_usd / (check.amount_src / exchange_rate[complete])).between(0.99, 1.01)
print(f"amount_usd == src / rate  on {pct_div.mean():.1%} of rows")

amount_usd == src * rate  on 11.5% of rows
amount_usd == src / rate  on 8.2% of rows


In [352]:
# DISCOVERY: if NOT the rate column, what DOES convert src -> usd correctly?
# Idea: amount_usd / amount_src might be a FIXED factor per source_currency.

factor = check.amount_usd / check.amount_src
summary = pd.DataFrame({
    "count": factor.groupby(check.source_currency).count(),
    "median": factor.groupby(check.source_currency).median(),
    "std": factor.groupby(check.source_currency).std(),
}).round(4)
print(summary)

                 count  median     std
source_currency                       
CAD               1191    0.74  0.1596
GBP               2045    1.25  0.2521
USD               7664     1.0  0.1792


In [353]:
 
factor_map = df_raw.source_currency.map(summary['median'])
healthy_rows = df_raw.amount_src.gt(0) & df_raw.amount_usd.notna()
ratio = df_raw.loc[healthy_rows, "amount_usd"] / (df_raw.loc[healthy_rows, "amount_src"] * factor_map[healthy_rows])
print(f"ratio median: {ratio.median():.6f}  (1.0 = conversion law holds)")
print(f"rows within 0.99-1.01: {ratio.between(0.99, 1.01).mean():.1%}")

ratio median: 1.000000  (1.0 = conversion law holds)
rows within 0.99-1.01: 100.0%


In [354]:
print(df_raw.amount_usd.isnull().sum())

300


In [355]:
df_raw.amount_usd[df_raw.amount_usd<0]

Series([], Name: amount_usd, dtype: float64)

In [356]:
# DISCOVERY: are the negative amount_src rows the SAME batch as the giant fees / negative velocity?
neg = df_raw.amount_src.lt(0)
big_fee = df_raw.fee.ge(9999)
neg_velocity = df_raw.txn_velocity_1h.lt(0)

print(f"negative amount_src:      {int(neg.sum())}")
print(f"  also fee >= 9999.99:    {int((neg & big_fee).sum())}")
print(f"  also velocity < 0:      {int((neg & neg_velocity).sum())}")
print(f"  all three overlap:      {int((neg & big_fee & neg_velocity).sum())}")


negative amount_src:      100
  also fee >= 9999.99:    100
  also velocity < 0:      100
  all three overlap:      100


*observation*: 
every negative amount belongs to the corrupt batch.
mask these (set to NaN), do NOT drop the rows: is_fraud lives on the rows.

In [357]:
# flag the corrupt batch, then mask the bad values to NaN (keep the rows).
df_raw["corrupt_record"] = (
    (df_raw.fee < 0) | (df_raw.fee > 100)
    | (df_raw.txn_velocity_1h < 0)
    | (df_raw.ip_risk_score > 1.0)
    | (df_raw.device_trust_score < 0)
    | df_raw.amount_src.lt(0)
)
print(f"corrupt rows flagged: {int(df_raw['corrupt_record'].sum())}")

df_raw.fee = df_raw.fee.mask((df_raw.fee < 0) | (df_raw.fee > 100))
df_raw.txn_velocity_1h = df_raw.txn_velocity_1h.mask(df_raw.txn_velocity_1h < 0)
df_raw.ip_risk_score = df_raw.ip_risk_score.mask(df_raw.ip_risk_score > 1.0)
df_raw.device_trust_score = df_raw.device_trust_score.mask(df_raw.device_trust_score < 0)
df_raw.amount_src = df_raw.amount_src.mask(df_raw.amount_src < 0)

print("fee NaN:", df_raw.fee.isna().sum(), "| velocity NaN:", df_raw.txn_velocity_1h.isna().sum(),
      "| ip NaN:", df_raw.ip_risk_score.isna().sum(), "| device NaN:", df_raw.device_trust_score.isna().sum(),
      "| src NaN:", df_raw.amount_src.isna().sum())

corrupt rows flagged: 404
fee NaN: 694 | velocity NaN: 200 | ip NaN: 200 | device NaN: 490 | src NaN: 100


In [358]:
# Impute the missing amount_usd with the EXACT conversion factor.
impute = df_raw.amount_usd.isna() & df_raw.amount_src.notna() & factor_map.notna()
df_raw.loc[impute, "amount_usd"] = df_raw.loc[impute, "amount_src"] * factor_map[impute]
print(f"amount_usd imputed: {int(impute.sum())} | remaining NaN: {int(df_raw.amount_usd.isna().sum())}")
print(f"negative amount_usd remaining: {int((df_raw.amount_usd < 0).sum())}")

amount_usd imputed: 297 | remaining NaN: 3
negative amount_usd remaining: 0


In [359]:
# inspect rows  for ip_address, home_country and ip_country
df_raw.loc[df_raw.ip_address.notna() | df_raw.ip_country.notna(), ['ip_address', 'home_country', 'ip_country']].head(20)

,ip_address,home_country,ip_country
0,126.36.182.218,US,unknown
1,177.207.86.226,US,CA
3,196.112.130.248,UK,UK
4,235.95.203.197,US,CA
5,218.154.47.179,US,US
6,178.148.100.236,US,CA
7,143.246.49.75,CA,CA
8,147.225.217.2,US,US
9,74.65.212.19,UK,UK
10,77.83.24.35,US,US


In [360]:
# inspect missing values for home_country and ip_country
missing_ip = df_raw.ip_country.isna()
missing_home = df_raw.home_country.isna()
print(f"ip_country missing: {int(missing_ip.sum())}")
print(f"home_country missing: {int(missing_home.sum())}")


ip_country missing: 296
home_country missing: 0


obeservation : home coutry does pick unknown 

In [361]:
# clean ip_country and home_country columns
print(f"Unique ip_country values: {df_raw.ip_country.unique()}")
print()
print(f"Unique home_country values: {df_raw.home_country.unique()}")

Unique ip_country values: <StringArray>
['unknown', 'CA', nan, 'UK', 'US', ' US  ', ' UK  ', ' CA  ', 'NAN', ' nan  ']
Length: 10, dtype: str

Unique home_country values: <StringArray>
['US', 'UK', 'CA', ' US  ', ' CA  ', 'unknown', ' UK  ']
Length: 7, dtype: str


In [362]:
df_raw.ip_country = df_raw.ip_country.str.lower().str.strip()
df_raw.home_country = df_raw.home_country.str.lower().str.strip()
print(f"Unique ip_country values after cleaning: {df_raw.ip_country.unique()}")
print()
print(f"Unique home_country values after cleaning: {df_raw.home_country.unique()}")

Unique ip_country values after cleaning: <StringArray>
['unknown', 'ca', nan, 'uk', 'us', 'nan']
Length: 6, dtype: str

Unique home_country values after cleaning: <StringArray>
['us', 'uk', 'ca', 'unknown']
Length: 4, dtype: str


In [363]:
df_raw.ip_country = df_raw.ip_country.replace({"nan":np.nan,"unknown":np.nan})
df_raw.home_country = df_raw.home_country.replace({"unknown":np.nan})
print(f"Unique ip_country values after cleaning: {df_raw.ip_country.unique()}")
print()
print(f"Unique home_country values after cleaning: {df_raw.home_country.unique()}")


Unique ip_country values after cleaning: <StringArray>
[nan, 'ca', 'uk', 'us']
Length: 4, dtype: str

Unique home_country values after cleaning: <StringArray>
['us', 'uk', 'ca', nan]
Length: 4, dtype: str


Keep ip_country NaN; the record_incomplete flag will mark these rows.


In [364]:
df_raw.kyc_tier.unique()

<StringArray>
[    'standrd',    'standard',           nan,         'low',    'enhanced',
    'ENHANCED', ' enhanced  ', ' standard  ',    'STANDARD',     'unknown',
         'LOW',      ' low  ',     'enhancd',         'NAN',      ' nan  ']
Length: 15, dtype: str

In [365]:
df_raw.channel.unique()

<StringArray>
[  'mobille',       'web',    'mobile',    ' web  ', ' mobile  ',       'ATM',
    'MOBILE',      'weeb',       'WEB',   'unknown',       'ATm',    ' ATM  ']
Length: 12, dtype: str

In [366]:
# fix kyc_tier spelling and casing values to 'standard' 'enhanced', 'low' and 'nan'-> NaN
df_raw.kyc_tier = df_raw.kyc_tier.str.strip().str.lower()
df_raw.kyc_tier = df_raw.kyc_tier.replace('nan', np.nan)

In [367]:
df_raw.kyc_tier.unique()

<StringArray>
['standrd', 'standard', nan, 'low', 'enhanced', 'unknown', 'enhancd']
Length: 7, dtype: str

In [368]:
# replace misspelled kyc_tier values with correct ones
df_raw.kyc_tier = df_raw.kyc_tier.replace({'standrd': "standard", 'enhancd': "enhanced", 'unknown': np.nan})
df_raw.kyc_tier = df_raw.kyc_tier.astype("string")
df_raw.kyc_tier.unique()

<StringArray>
['standard', <NA>, 'low', 'enhanced']
Length: 4, dtype: string

In [369]:
df_raw.channel = df_raw.channel.str.strip().str.lower()
df_raw.channel.unique()

<StringArray>
['mobille', 'web', 'mobile', 'atm', 'weeb', 'unknown']
Length: 6, dtype: str

In [370]:
df_raw.channel = df_raw.channel.replace({"mobille":"mobile","unknown":np.nan,"weeb":"web"})
df_raw.channel = df_raw.channel.astype("string")
df_raw.channel.unique()

<StringArray>
['mobile', 'web', 'atm', <NA>]
Length: 4, dtype: string

In [371]:
#which columns are missing, in what counts
missing_counts = df_raw.isnull().sum()
missing_summary = (
    missing_counts[missing_counts > 0]
    .to_frame(name="missing_count")
    .assign(missing_pct= lambda x: x.missing_count / len(df_raw) * 100).round(1)
    .sort_values(by="missing_pct", ascending=False)
)

print("Missing summary:")
print(missing_summary)

Missing summary:
                    missing_count  missing_pct
fee                           694          6.2
device_trust_score            490          4.4
ip_country                    331          3.0
kyc_tier                      329          2.9
ip_address                    300          2.7
ip_risk_score                 200          1.8
txn_velocity_1h               200          1.8
amount_src                    100          0.9
timestamp                      60          0.5
home_country                   32          0.3
channel                        37          0.3
amount_usd                      3          0.0


In [372]:
# Analyze missingness patterns
patterns = df_raw.isnull().dot(df_raw.columns + '|')  # missingness pattern as string
pattern_counts = patterns.value_counts()
print(pattern_counts.head(10))

                                                                    10342
fee|ip_address|ip_country|kyc_tier|device_trust_score|                289
fee|                                                                  201
amount_src|fee|ip_risk_score|device_trust_score|txn_velocity_1h|       97
fee|ip_risk_score|device_trust_score|txn_velocity_1h|                  93
timestamp|                                                             57
channel|                                                               25
ip_country|                                                            24
kyc_tier|                                                              20
home_country|                                                          20
Name: count, dtype: int64


In [373]:
# Clustering check: do missing values overlap in the same rows?
# Purpose: when columns go missing together, they often share a root cause
# (e.g., "field not collected" batches, api failures, KYC tier gaps).
# This helps decide joint imputation vs. flag-as-feature.

missing_corr = df_raw.isnull().corr().abs()  # pairwise |r| of missingness indicators

# Build a clean long-format table of high-correlation pairs
pairs = missing_corr.where(missing_corr > 0.8).stack().reset_index()
pairs.columns = ['col_a', 'col_b', 'corr']

# Remove self-pairs and symmetric duplicates, sort by strength
pairs = pairs[
    (pairs.col_a != pairs.col_b) &
    (pairs.col_a < pairs.col_b)  # keeps only one of (a,b) / (b,a)
].sort_values('corr', ascending=False)

print("High missingness overlap (|r| > 0.8):")
if pairs.empty:
    print("  No column pairs with > 0.8 missingness correlation.")
else:
    print(pairs.head(25).to_string(index=False))

# Also show the count of rows missing multiple fields 
multi_null_counts = df_raw.isnull().sum(axis=1).value_counts().sort_index()
print("\nRows by count of missing fields:")
print(multi_null_counts.head(25))

High missingness overlap (|r| > 0.8):
             col_a                     col_b     corr
     ip_risk_score           txn_velocity_1h 1.000000
        ip_address                  kyc_tier 0.953639
        ip_address                ip_country 0.950666
        ip_country                  kyc_tier 0.915702
device_trust_score                       fee 0.832228
    transaction_id           txn_velocity_1h      NaN
    transaction_id          txn_velocity_24h      NaN
       customer_id            transaction_id      NaN
       customer_id                 timestamp      NaN
       customer_id              home_country      NaN
       customer_id           source_currency      NaN
       customer_id             dest_currency      NaN
       customer_id                       fee      NaN
       customer_id exchange_rate_src_to_dest      NaN
       customer_id                 device_id      NaN
       customer_id                new_device      NaN
       customer_id                ip_address

In [374]:
# View rows where BOTH device_trust_score AND fee are missing
# df[df['device_trust_score'].isnull() & df['fee'].isnull()].head(10)

In [375]:
# View specific columns for rows missing kyc_tier
# df[df['kyc_tier'].isnull()][['customer_id', 'channel', 'account_age_days', 'is_fraud']].head(10)

In [376]:
# Compare fraud rates: Missing vs. Present for a specific feature
# target_col = 'device_trust_score'

# fraud_rate_missing = df[df[target_col].isnull()]['is_fraud'].mean()
# fraud_rate_present = df[df[target_col].notnull()]['is_fraud'].mean()

# print(f"Fraud rate when {target_col} IS MISSING:    {fraud_rate_missing:.4f}")
# print(f"Fraud rate when {target_col} IS PRESENT:    {fraud_rate_present:.4f}")

In [377]:
# Find rows with 3 or more missing fields at the same time
# rows_with_many_nulls = df[df.isnull().sum(axis=1) >= 3]

# print(f"Total transactions with 3+ missing fields: {len(rows_with_many_nulls)}")
# print(f"Fraud rate in multi-null transactions:     {rows_with_many_nulls['is_fraud'].mean():.4f}")

# # Display sample multi-null records
# rows_with_many_nulls.head()

In [378]:
#Feature engineer record incompleteness for signal 
#commentry: there doesn't seem to be randomness in the missing gaps so i will preserve record incompleteness as a potential signal. 
#Note: I think i need to write a compute during inference for this feature as well since its not a default field from the transaction finlora will get. 
#compute AFTER all masking/imputation for any incomplete records
df_raw["timestamp_missing"] = df_raw.timestamp.isna()
df_raw["record_incomplete"] = df_raw.isna().any(axis=1)
print(f"timestamp_missing rows: {int(df_raw['timestamp_missing'].sum())}")
print(f"record_incomplete rows: {int(df_raw['record_incomplete'].sum())}")

timestamp_missing rows: 60
record_incomplete rows: 858


In [379]:
# df.info()